# EXP-011 dev-small 확인: 다중 크기 청킹(512 + 256)

EXP-009 기준(512 청크)과 EXP-011(dense 목록에 256 청크를 더함)의 dev-small 지표와 질의별 순위 변화, 추가 인덱스 용량을 본다. 판정에 쓰지 않는다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from rag.eval.metrics import gold_ranks, load_cfg, load_gold, read_run

In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
INDEX = ROOT / 'data' / 'index'
RUN_FILES = {
    'dense 512': 'snowflake-ko-fixed-512-64-dev-small.csv',
    'dense 512+256': 'snowflake-ko-fixed-512-64-dense-ms256-dev-small.csv',
    'EXP-009 hybrid': 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-dev-small.csv',
    'EXP-011 hybrid': 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-dev-small.csv',
}
INDEX_DIRS = ['snowflake-ko-fixed-512-64', 'snowflake-ko-fixed-256-32']

## 지표

In [3]:
gold = load_gold(load_cfg(), 'dev-small')
ranks = {name: gold_ranks(read_run(RUNS / f), gold, 10) for name, f in RUN_FILES.items()}
print(len(gold), 'queries')
table = {}
for name, r in ranks.items():
    row = {f'recall@{k}': (r <= k).mean() for k in (1, 5, 10)}
    row['mrr@10'] = (1 / r).where(r <= 10, 0).mean()
    table[name] = row
print(pd.DataFrame(table).round(4))

1039 queries
           dense 512  dense 512+256  EXP-009 hybrid  EXP-011 hybrid
recall@1      0.8749         0.8980          0.9105          0.9259
recall@5      0.9702         0.9750          0.9856          0.9865
recall@10     0.9846         0.9856          0.9885          0.9894
mrr@10        0.9180         0.9320          0.9433          0.9525


## 기준 대비 질의별 이동

In [4]:
for base, cand in [('dense 512', 'dense 512+256'), ('EXP-009 hybrid', 'EXP-011 hybrid')]:
    b, c = ranks[base], ranks[cand]
    for k in (1, 5):
        gain = int(((b.isna() | (b > k)) & (c <= k)).sum())
        loss = int(((b <= k) & (c.isna() | (c > k))).sum())
        print(f'{cand} vs {base} top{k}: 진입 {gain}, 이탈 {loss}')

dense 512+256 vs dense 512 top1: 진입 35, 이탈 11
dense 512+256 vs dense 512 top5: 진입 8, 이탈 3
EXP-011 hybrid vs EXP-009 hybrid top1: 진입 26, 이탈 10
EXP-011 hybrid vs EXP-009 hybrid top5: 진입 2, 이탈 1


## 인덱스 용량 (dev-small)

In [5]:
def du(p):
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())

rows = []
for name in INDEX_DIRS:
    d = INDEX / name / 'dev-small'
    n = sum(1 for _ in (d / 'chunks.jsonl').open(encoding='utf-8'))
    rows.append({'index': name, 'chunks': n, 'emb_MB': du(d / 'emb') / 1e6,
                 'chunks_MB': (d / 'chunks.jsonl').stat().st_size / 1e6})
sizes = pd.DataFrame(rows).set_index('index')
print(sizes.round(1))
print('256 / 512 청크 수 비율', round(sizes.chunks.iloc[1] / sizes.chunks.iloc[0], 3))

                           chunks  emb_MB  chunks_MB
index                                               
snowflake-ko-fixed-512-64    2995    12.3        6.7
snowflake-ko-fixed-256-32    5590    22.9        7.1
256 / 512 청크 수 비율 1.866
